# Session 13 · Case study: vocabulary and document-term matrix

**Block 1 practice.** Build the vocabulary of the descriptions of goods and inspect the document-term matrix.

You will

1. read one decision in its original language next to its English keywords and the English text of its heading,
2. count tokens and types per language in the 50,000-decision sample,
3. build a document-term matrix with `CountVectorizer` and measure its size and sparsity,
4. look at the most frequent and the rarest words (Zipf's law),
5. measure how preprocessing (lower-casing, stop words, German stemming, character n-grams) changes the vocabulary.

Theory: [01-text-as-data.md](../theory/01-text-as-data.md). Data: `case-study/data/train_sample.parquet` (prepare it with `case-study/prepare_data.py`).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
texts = decisions["description"]                     # one document = one description of goods
print(decisions.shape, decisions["heading"].nunique(), "headings")

## 1. Reading a decision you cannot read

The English **keywords** (assigned by customs) and the English **heading description** (from the nomenclature) help to understand a description in any language. They are reading aids only: the test decisions have neither keywords nor heading, so they are never model inputs.

In [ ]:
pd.set_option("display.max_colwidth", 300)
for language in ["de", "fr", "pl"]:
    row = decisions[decisions["language"] == language].iloc[5]
    print(f"[{language}] heading {row['heading']}: {nomenclature.loc[row['heading'], 'heading_description'][:90]}")
    print("  keywords:   ", row["keywords"])
    print("  description:", row["description"][:300].replace("\r", " ").replace("\n", " "), "\n")

## 2. Tokens, types and languages

A first, naive tokenisation: lower-case and split at white space.

In [ ]:
tokens = texts.str.lower().str.split()
n_tokens = tokens.str.len()
print("tokens in total:", int(n_tokens.sum()))
print(n_tokens.groupby(decisions["language"]).median().sort_values().tail(6))

types = set()
for t in tokens:
    types.update(t)
print("types (distinct tokens):", len(types))
print(decisions["language"].value_counts(normalize=True).head(8).round(3).to_dict())

**Question.** Why is the naive number of types larger than the vocabulary that `CountVectorizer` finds below? Look at a few types that contain punctuation.

In [ ]:
print(sorted(t for t in types if t.startswith("kunststoff"))[:15])

## 3. The document-term matrix

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()                       # lower-case, default token pattern
D = cv.fit_transform(texts)                  # scipy sparse CSR matrix
vocab = cv.get_feature_names_out()
print("shape:", D.shape)
print("non-zero cells:", D.nnz, f"({D.nnz / (D.shape[0] * D.shape[1]):.5f} of all cells)")
print("distinct words per description (mean):", round(D.nnz / D.shape[0], 1))
sparse_mb = (D.data.nbytes + D.indices.nbytes + D.indptr.nbytes) / 1e6
print(f"memory: {sparse_mb:.1f} MB sparse vs {D.shape[0] * D.shape[1] * 8 / 1e9:.1f} GB dense")

Inspect one description and its row of the matrix. Only the non-zero entries are stored.

In [ ]:
i = 7
print(texts.iloc[i][:300])
row = D[i]
print(pd.Series(row.data, index=vocab[row.indices]).sort_values(ascending=False).head(15))

## 4. Frequent and rare words (Zipf's law)

In [ ]:
import matplotlib.pyplot as plt

counts = pd.Series(np.asarray(D.sum(axis=0)).ravel(), index=vocab).sort_values(ascending=False)
doc_freq = pd.Series(np.bincount(D.indices, minlength=D.shape[1]), index=vocab)
print("most frequent:", counts.head(12).index.tolist())
print("share of words that occur only once:", round((counts == 1).mean(), 2))
print("words in at least 5 descriptions:", int((doc_freq >= 5).sum()))
print("share of pure numbers among the types:", round(counts.index.str.fullmatch(r"\d+").mean(), 3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(np.arange(1, len(counts) + 1), counts.to_numpy())
ax.set_xlabel("rank of word (log)")
ax.set_ylabel("frequency (log)")
ax.set_title("Zipf's law in the description sample")
plt.show()

**Question.** The most frequent words are German function words. Why? What would happen to the top of this list if you used only French decisions?

## 5. Preprocessing and the size of the vocabulary

In [ ]:
from nltk.stem.snowball import SnowballStemmer

german = SnowballStemmer("german")
base = CountVectorizer().build_analyzer()          # the default tokeniser as a function


def german_stem_analyzer(doc):
    return [german.stem(t) for t in base(doc)]


settings = {
    "default (lower-case)": CountVectorizer(),
    "no lower-casing": CountVectorizer(lowercase=False),
    "English stop words removed": CountVectorizer(stop_words="english"),
    "min_df=5": CountVectorizer(min_df=5),
    "German Snowball stemming (all languages!)": CountVectorizer(analyzer=german_stem_analyzer),
    "word unigrams + bigrams, min_df=3": CountVectorizer(ngram_range=(1, 2), min_df=3),
    "character n-grams 3-5 (char_wb), min_df=3": CountVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3),
}
sample = texts.sample(10_000, random_state=0)      # a subsample keeps stemming fast
rows = []
for name, vec in settings.items():
    M = vec.fit_transform(sample)
    rows.append({"setting": name, "vocabulary": M.shape[1], "non-zeros per document": round(M.nnz / M.shape[0])})
pd.DataFrame(rows)

## 6. Compounds and character n-grams

In [ ]:
compounds = [w for w in vocab if w.endswith("spielzeug")]
print(len(compounds), "words end in 'spielzeug':", compounds[:12])
grams = CountVectorizer(analyzer="char_wb", ngram_range=(4, 4)).build_analyzer()
print(sorted(set(grams("kunststoffspielzeug")) & set(grams("spielzeug"))))

## Your turn

1. Which setting removes the most columns? Which one adds the most? Why does the English stop-word list change almost nothing?
2. Applying a German stemmer to all languages is wrong for French or Polish. Write an analyzer that stems only German descriptions (hint: pass the language with the text, or build two vectorisers) and compare the vocabulary size.
3. Find the five most frequent words in French descriptions and in Polish descriptions (`decisions["language"] == "fr"`). Would a stop-word list per language be worth the effort?
4. Optional: how many descriptions contain the placeholder `<CODE>`? What token does the default tokeniser make of it?

In [ ]:
# your code here
print(decisions["description"].str.contains("<CODE>").mean().round(3))